In [1]:
#%pip install paramiko

StatementMeta(, , -1, Cancelled, , Cancelled, True)


# **Tables array**


In [1]:
TECH_SCD2_COLS = ["hash_value", "Start_Date", "End_Date", "IsCurrent"]

tables = [
  {
    "name": "dimaccountgrouphierarchy__0001",
    "type": "SCD2",
    "business_key": ["AccountGroupHierarchyKey"],
    "exclude_hash_cols": [
        "RecordCreatedDate", "RecordModifiedDate",
        *TECH_SCD2_COLS
    ]
  },
  {
    "name": "dimclassificationlist__0001",
    "type": "SCD2",
    "business_key": ["ClassificationListKey"],
    "exclude_hash_cols": [
        "RecordCreatedDate", "RecordModifiedDate",
        *TECH_SCD2_COLS
    ]
  },
  {
    "name": "dimleadmanagement__0001",
    "type": "SCD2",
    "business_key": ["LeadManagementKey"],
    "exclude_hash_cols": [
        "RecordCreatedDate", "RecordModifiedDate",
        *TECH_SCD2_COLS
    ]
  },
  {
    "name": "dimleadmetrics__0001",
    "type": "SCD2",
    "business_key": ["LeadMetricsKey"],
    "exclude_hash_cols": [
        "RecordCreatedDate", "RecordModifiedDate", "Date","Processed_Time",
        *TECH_SCD2_COLS
    ]
  },
  {
    "name": "dimleaseattributes__0001",
    "type": "SCD2",
    "business_key": ["LeaseAttributesKey"],
    "exclude_hash_cols": [
        "RecordCreatedDate", "RecordModifiedDate","CDSExtractDate",
        *TECH_SCD2_COLS
    ]
  },
  {
    "name": "dimresidentactivitylog__0001",
    "type": "SCD2",
    "business_key": ["ResidentActivityLogkey"],
    "include_hash_cols": [
        "ResidentActivityLogkey", "PropertyKey","ResidentKey","CodeActivityTypeCode"
    ]
  },
  {
    "name": "dimleaseexpiration__0001",
    "type": "SCD2",
    "business_key": ["LeaseExpirationKey"],
    "exclude_hash_cols": [
        "RecordCreatedDate", "RecordModifiedDate",
        *TECH_SCD2_COLS
    ]
  },
  {
    "name": "dimmakereadyrequest__0001",
    "type": "SCD2",
    "business_key": ["MakeReadyRequestKey"],
    "exclude_hash_cols": [
        "RecordCreatedDate", "RecordModifiedDate",
        "osl_ExtractDate", "CreateDate","osl_ExtractDate","UnitKey",
        *TECH_SCD2_COLS
    ]
  },
  {
    "name": "dimproperty__0001",
    "type": "SCD2",
    "business_key": ["PropertyKey"],
    "include_hash_cols": [
        "PropertyKey", "OrganizationKey", "PropertyName", "PropertyAddress1"
    ]
  },
  {
    "name": "dimresidentmember__0001",
    "type": "SCD2",
    "business_key": ["ResidentMemberKey"],
    "exclude_hash_cols": [
        "RecordCreatedDate", "RecordModifiedDate", "CDSExtractDate",
        *TECH_SCD2_COLS
    ]
  },
  {
    "name": "dimscreeningapplicant__0001",
    "type": "SCD2",
    "business_key": ["ApplicantKey"],
    "exclude_hash_cols": [
        "RecordCreatedDate", "RecordModifiedDate", "RowStartDate","BirthDate","ApplicationDate",
        *TECH_SCD2_COLS
    ]
  },
  {
    "name": "dimservicerequest__0001",
    "type": "SCD2",
    "business_key": ["ServiceRequestKey"],
    "exclude_hash_cols": [
        "RecordCreatedDate", "RecordModifiedDate",
        "RequestTime", "CompleteTime", "UnitKey","WorkingDays","mrrrMakeReadyStartDay","MRWorkingDays","mrrrMoveOutDate",
        *TECH_SCD2_COLS
    ]
  },
  {
    "name": "factoperationalkpi__0001",
    "type": "SCD2",
    "business_key": ["PropertyKey"],
    "exclude_hash_cols": [
        "RecordCreatedDate", "RecordModifiedDate",
        *TECH_SCD2_COLS
    ]
  },
  {
    "name": "factaccountgrouphierarchydetail__0001",
    "type": "SCD2",
    "business_key": ["AccountGroupHierarchyDetailKey"],
    "exclude_hash_cols": [
        "RecordCreatedDate", "RecordModifiedDate",
        *TECH_SCD2_COLS
    ]
  },
  {
    "name": "factglsummary__0001",
    "type": "SCD2",
    "business_key": ["Tablekey"],
    "exclude_hash_cols": [
        "RecordCreatedDate", "RecordModifiedDate",
        *TECH_SCD2_COLS
    ]
  },
  {
    "name": "factpropertystatisticsasofdate__0001",
    "type": "SCD2",
    "business_key": ["PropertyKey","PostDate"],
    "exclude_hash_cols": [
        "RecordCreatedDate", "RecordModifiedDate",
        *TECH_SCD2_COLS
    ]
  },
  {
    "name": "factscreeningapplicant__0001",
    "type": "SCD2",
    "business_key": ["ApplicantKey"],
    "exclude_hash_cols": [
        "RecordCreatedDate", "RecordModifiedDate",
        *TECH_SCD2_COLS
    ]
  }
   



]

StatementMeta(, 80d2bf95-7889-468b-b17f-413f3a3bf9f5, 3, Finished, Available, Finished, False)


# **SFTP CSV PULL ------> Files**


In [10]:
# ======================= ONE CELL: RealPage SFTP -> Lakehouse Files (FABRIC HARD RESET) =======================
# ✅ ALWAYS completely DELETES Files/RealPageDaily
# ✅ Then downloads everything again
# ✅ Cleans file names keeping only the part after dbo__
# ==============================================================================================================

import os, re, stat, pathlib, zoneinfo, datetime as dt, paramiko
import notebookutils as nb

fs = nb.fs

# -------------------- SETTINGS --------------------
HOST, PORT = "bimft.realpage.com", 22
USER       = "EXT-HERITAGEHILLPROPMGMT_BI"
PWD        = "BJEEod^ykvA9Rm"

REMOTE_DIR = "/BI_Download"
TMP_DIR    = pathlib.Path("/tmp/realpage")
LAKE_DIR   = "Files/RealPageDaily"

tz    = zoneinfo.ZoneInfo("America/New_York")
token = dt.datetime.now(tz).strftime("%Y%m%d")
rx    = re.compile(rf"{token}.*\.csv$", re.I)

print("📅 Token:", token)
print("📡 Remote:", REMOTE_DIR)
print("📁 Lakehouse:", LAKE_DIR)

# ================== HARD DELETE FOLDER COMPLETELY ==================
try:
    fs.rm(LAKE_DIR, True)   # Borra TODO recursivamente
    print(f"Folder deleted: {LAKE_DIR}")
except Exception as e:
    print("Folder no existía, seguimos...")

# Recrear carpeta limpia
fs.mkdirs(LAKE_DIR)
print(f"Folder recreated: {LAKE_DIR}")

# -------------------- PREP TMP --------------------
TMP_DIR.mkdir(exist_ok=True, parents=True)

# -------------------- SFTP COPY --------------------
copied = []

with paramiko.Transport((HOST, PORT)) as tr:
    tr.connect(username=USER, password=PWD)
    print("✓ Connected to SFTP server")

    with paramiko.SFTPClient.from_transport(tr) as sftp:

        try:
            entries = sftp.listdir_attr(REMOTE_DIR)
        except FileNotFoundError:
            REMOTE_DIR = REMOTE_DIR.lstrip("/")
            entries    = sftp.listdir_attr(REMOTE_DIR)

        for e in entries:
            if stat.S_ISREG(e.st_mode) and rx.search(e.filename):
                remote = f"{REMOTE_DIR}/{e.filename}"
                local  = TMP_DIR / e.filename
                lake   = f"{LAKE_DIR}/{e.filename}"

                sftp.get(remote, str(local))
                fs.fastcp(f"file:{local}", lake, True)
                local.unlink(missing_ok=True)

                copied.append(e.filename)

print(f"✓ Copied {len(copied)} file(s): {copied}")

# ================== 🧼 RENAME CLEAN (dbo__) ==================
regex = re.compile(r".*dbo__(.+)", re.I)

renamed = []
for entry in fs.ls(LAKE_DIR):
    src_path = entry.path
    fname    = src_path.split("/")[-1]

    m = regex.match(fname)
    if m:
        new_fname = m.group(1)
        dst_path  = f"{LAKE_DIR}/{new_fname}"
        fs.mv(src_path, dst_path, True, True)
        renamed.append((fname, new_fname))

print("✓ Renamed", len(renamed), "file(s):")
for old, new in renamed:
    print(f"   {old}  →  {new}")

StatementMeta(, 837beadd-fd7c-4962-8796-24ec86d89df3, 12, Finished, Available, Finished, False)

📅 Token: 20260305
📡 Remote: /BI_Download
📁 Lakehouse: Files/RealPageDaily
🧹 Folder deleted: Files/RealPageDaily
✓ Folder recreated: Files/RealPageDaily
✓ Connected to SFTP server
✓ Copied 440 file(s): ['RA3712076__20260305__dbo__ClientTags__0001.csv', 'RA3712076__20260305__dbo__DimAcademicCalendar__0001.csv', 'RA3712076__20260305__dbo__DimAccountGroupHierarchy__0001.csv', 'RA3712076__20260305__dbo__DimAccountingBatch__0001.csv', 'RA3712076__20260305__dbo__DimAccountingCalendar__0001.csv', 'RA3712076__20260305__dbo__DimAccountingCustomField__0001.csv', 'RA3712076__20260305__dbo__DimAccountingDepartmentCustomField__0001.csv', 'RA3712076__20260305__dbo__DimAccountingDepartment__0001.csv', 'RA3712076__20260305__dbo__DimAccountingEntityContact__0001.csv', 'RA3712076__20260305__dbo__DimAccountingEntryCustomField__0001.csv', 'RA3712076__20260305__dbo__DimAccountingRecordCustomField__0001.csv', 'RA3712076__20260305__dbo__DimAccountingRP_EntityMRCodeDetail__0001.csv', 'RA3712076__20260305__dbo_

## **Replace Historic Tables**

In [2]:
import re
from datetime import date
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, DateType, TimestampType
from pyspark.sql.window import Window
import notebookutils as nb

fs = nb.fs

FILES_DIR_FS = "Files/RealPageDaily"   # solo para listar

HASH_COL = "hash_value"
START_COL = "Start_Date"
END_COL   = "End_Date"
ISCURRENT_COL = "IsCurrent"

MAX_END = "12/31/9999"
TODAY = date.today()
TODAY_STR = f"{TODAY.month}/{TODAY.day}/{TODAY.year}"

spark.conf.set("spark.sql.legacy.timeParserPolicy", "CORRECTED")

DATE_PATTERNS = [
    "M/d/yyyy","MM/dd/yyyy","M/d/yy","MM/dd/yy",
    "yyyy-MM-dd","yyyy/MM/dd","yyyyMMdd",
    "dd-MMM-yyyy","d-MMM-yyyy","MMM d, yyyy","MMMM d, yyyy"
]
TS_PATTERNS = [
    "M/d/yyyy h:mm:ss a","MM/dd/yyyy h:mm:ss a",
    "M/d/yyyy hh:mm:ss a","MM/dd/yyyy hh:mm:ss a",
    "M/d/yyyy H:mm:ss","MM/dd/yyyy H:mm:ss",
    "M/d/yyyy HH:mm:ss","MM/dd/yyyy HH:mm:ss",
    "yyyy-MM-dd HH:mm:ss","yyyy-MM-dd'T'HH:mm:ss","yyyy-MM-dd'T'HH:mm:ss.SSS"
]

def _parsed_date_expr(col_trim):
    parsed_ts = F.coalesce(*[F.to_timestamp(col_trim, p) for p in TS_PATTERNS])
    parsed_d  = F.coalesce(*[F.to_date(col_trim, p) for p in DATE_PATTERNS])
    return F.coalesce(F.to_date(parsed_ts), parsed_d)

def standardize_date_columns(df, sample_rows=1000, success_threshold=0.90, min_non_null=10):
    out = df
    sample = df.limit(sample_rows)

    for field in df.schema.fields:
        c = field.name
        dtp = field.dataType

        if isinstance(dtp, (DateType, TimestampType)):
            out = out.withColumn(c, F.date_format(F.to_date(F.col(c)), "M/dd/yyyy"))
            continue

        if isinstance(dtp, StringType):
            col_trim = F.trim(F.col(c))
            parsed_date = _parsed_date_expr(col_trim)

            stats = (
                sample.select(
                    F.sum(F.when(col_trim.isNotNull() & (F.length(col_trim) > 0), 1).otherwise(0)).alias("non_null"),
                    F.sum(F.when((col_trim.isNotNull() & (F.length(col_trim) > 0)) & parsed_date.isNotNull(), 1).otherwise(0)).alias("parsed_ok")
                ).collect()[0]
            )

            non_null = int(stats["non_null"] or 0)
            parsed_ok = int(stats["parsed_ok"] or 0)
            rate = (parsed_ok / non_null) if non_null > 0 else 0.0

            if non_null >= min_non_null and rate >= success_threshold:
                out = out.withColumn(c, F.date_format(parsed_date, "M/dd/yyyy"))

    return out

def _norm_key(s: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", (s or "").lower().strip())

def normalize_excludes(df, exclude_cols):
    cols_map = {_norm_key(c): c for c in df.columns}
    out = []
    for x in (exclude_cols or []):
        k = _norm_key(str(x))
        if k in cols_map:
            out.append(cols_map[k])
    return out

# ✅ CHANGED: supports include_cols (whitelist) OR exclude_cols (blacklist)
def add_hash_like_scd2(df, exclude_cols=None, include_cols=None):
    if include_cols:
        include_set = set(include_cols)
        cols_for_hash = [c for c in df.columns if c in include_set]
        cols_for_hash = sorted(cols_for_hash, key=lambda x: x.lower())
    else:
        exclude_set = set((exclude_cols or []) + [HASH_COL, START_COL, END_COL, ISCURRENT_COL])
        cols_for_hash = [c for c in df.columns if c not in exclude_set]
        cols_for_hash = sorted(cols_for_hash, key=lambda x: x.lower())

    if not cols_for_hash:
        return df.withColumn(HASH_COL, F.sha2(F.lit(""), 256))

    exprs = [F.coalesce(F.trim(F.col(c).cast("string")), F.lit("")) for c in cols_for_hash]
    return df.withColumn(HASH_COL, F.sha2(F.concat_ws("||", *exprs), 256))

# ✅ NEW: resolves include_hash_cols vs exclude_hash_cols from config
def _resolve_hash_kwargs(df, cfg: dict) -> dict:
    incl_cfg = cfg.get("include_hash_cols")
    if incl_cfg:
        return {"include_cols": normalize_excludes(df, incl_cfg)}
    excl_cfg = cfg.get("exclude_hash_cols", [])
    return {"exclude_cols": normalize_excludes(df, excl_cfg)}

def _order_ts(df, colname):
    if colname not in df.columns:
        return None
    col_trim = F.trim(F.col(colname).cast("string"))
    d = _parsed_date_expr(col_trim)
    return F.to_timestamp(d)

def dedup_like_scd2(df, bk_cols):
    order_cols = ["RecordModifiedDate","RecordCreatedDate","CDSExtractDate","osl_ExtractDate","CreateDate","RowStartDate","ModifyDate","PostDate"]
    order_exprs = []
    for c in order_cols:
        ts_expr = _order_ts(df, c)
        if ts_expr is not None:
            order_exprs.append(ts_expr.desc_nulls_last())

    if HASH_COL in df.columns:
        order_exprs.append(F.col(HASH_COL).desc_nulls_last())
    if "SourceKey" in df.columns:
        order_exprs.append(F.col("SourceKey").desc_nulls_last())
    for c in bk_cols:
        if c in df.columns:
            order_exprs.append(F.col(c).cast("string").desc_nulls_last())

    if not order_exprs:
        return df.dropDuplicates(bk_cols)

    w = Window.partitionBy(*[F.col(c) for c in bk_cols]).orderBy(*order_exprs)
    return (df.withColumn("_rn", F.row_number().over(w))
              .where(F.col("_rn") == 1)
              .drop("_rn"))

def _canon_table(s: str) -> str:
    s = (s or "").strip()
    s = re.sub(r"\.csv$", "", s, flags=re.I)
    s = re.sub(r"^dim", "dim", s, flags=re.I)  # Dim vs DIM
    return s.lower()

# ✅ returns REAL abfss path from fs listing (this avoids the 400 HEAD on /lakehouse/default)
def find_existing_csv_abfss(table_name: str) -> str:
    target = _canon_table(table_name)
    entries = fs.ls(FILES_DIR_FS)

    files = [e for e in entries if (not e.isDir and e.name.lower().endswith(".csv"))]

    for e in files:
        if _canon_table(e.name) == target:
            return e.path  # ✅ ABFSS path

    hits = [e for e in files if target in _canon_table(e.name)]
    if hits:
        return hits[0].path

    raise Exception(f"No encontré CSV para {table_name} en {FILES_DIR_FS}")

# ======================= RUN (your array) =======================
CURRENT_DB = spark.sql("select current_database() as db").collect()[0]["db"]

failed = []
for cfg in tables:
    try:
        name = cfg["name"]
        bk   = cfg.get("business_key", [])
        excl_cfg = cfg.get("exclude_hash_cols", [])

        if not bk:
            raise Exception(f"{name}: business_key vacío")

        csv_path = find_existing_csv_abfss(name)  # ✅ ABFSS real

        df = (spark.read
              .option("header", True)
              .option("inferSchema", True)
              .csv(csv_path))

        df = standardize_date_columns(df)
        excl = normalize_excludes(df, excl_cfg)
        df = add_hash_like_scd2(df, excl)
        df = dedup_like_scd2(df, bk)

        df_out = (df
            .withColumn(START_COL, F.lit(TODAY_STR))
            .withColumn(END_COL, F.lit(MAX_END))
            .withColumn(ISCURRENT_COL, F.lit(1))
        )

        full_table = f"{CURRENT_DB}.{name}"

        (df_out.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(full_table))

        print(f"✅ OVERWRITE {full_table} | csv={csv_path}")

    except Exception as e:
        failed.append((cfg.get("name","unknown"), str(e)))
        print(f"❌ Falló {cfg.get('name','unknown')}: {str(e)[:300]}")

print("\n🎯 FINISH")
if failed:
    print("⚠ Tablas fallidas:")
    for n, err in failed:
        print("  •", n, "|", err[:300])
else:
    print("✅ Listo: usando ABFSS real (fs.ls) ya no pega el 400 de lakehouse/default.")

StatementMeta(, 80d2bf95-7889-468b-b17f-413f3a3bf9f5, 4, Finished, Available, Finished, False)

✅ OVERWRITE realpagebix__lah_df2.dimresidentactivitylog__0001 | csv=abfss://b13d87d7-076e-42df-a992-14dbd7f186f2@onelake.dfs.fabric.microsoft.com/cae086eb-7751-49db-87cd-e57f036d9d05/Files/RealPageDaily/DimResidentActivityLog__0001.csv

🎯 FINISH
✅ Listo: usando ABFSS real (fs.ls) ya no pega el 400 de lakehouse/default.


 ## **Get tables From Backup**

In [10]:
# ======================= FABRIC: Copy ALL CSVs from Backup Files -> Target Lakehouse Files/RealPageDaily
# ✅ Deletes everything inside target Files/RealPageDaily
# ✅ Copies all files from backup folder (CSV) to target folder
# ✅ Uses ABFSS for BOTH source and destination (fixes 400 Bad Request)

from notebookutils import fs

WORKSPACE_ID = "b13d87d7-076e-42df-a992-14dbd7f186f2"

# SOURCE (backup lakehouse)
SRC_LH_ID = "da429e12-11e6-46fb-9769-53af4ec87b21"
SRC_PATH  = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{SRC_LH_ID}/Files/Backup_asof_2025-12-31"

# DEST (target lakehouse where you want RealPageDaily)
DST_LH_ID = "cae086eb-7751-49db-87cd-e57f036d9d05"
DST_PATH  = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{DST_LH_ID}/Files/RealPageDaily"

print("SRC_PATH =", SRC_PATH)
print("DST_PATH =", DST_PATH)

def safe_ls(p):
    try:
        return fs.ls(p)
    except Exception:
        return []

def rm_all_inside(dir_path):
    fs.mkdirs(dir_path)
    items = safe_ls(dir_path)
    ok = fail = 0
    for it in items:
        try:
            fs.rm(it.path, True)
            ok += 1
        except Exception as e:
            fail += 1
            print("rm failed:", it.path, "|", str(e)[:200])
    return ok, fail

def copy_all_files(src_dir, dst_dir):
    fs.mkdirs(dst_dir)
    items = safe_ls(src_dir)
    if not items:
        raise RuntimeError("No pude listar el source. Revisá permisos/ruta.")

    ok = fail = 0
    for it in items:
        name = it.name.strip("/")
        src  = it.path
        dst  = f"{dst_dir}/{name}"
        try:
            # recurse=True (sirve si algún item es folder)
            fs.cp(src, dst, True)
            ok += 1
            if ok % 50 == 0:
                print(f"  ...copied {ok}")
        except Exception as e:
            fail += 1
            print("cp failed:", src, "->", dst, "|", str(e)[:220])
    return ok, fail

# 1) Clean destination
print("\n🧹 Cleaning destination ...")
del_ok, del_fail = rm_all_inside(DST_PATH)
print(f"Deleted entries: ok={del_ok}, fail={del_fail}")
print("Destination items now:", len(safe_ls(DST_PATH)))

# 2) Copy all
print("\n📦 Copying backup -> destination ...")
cp_ok, cp_fail = copy_all_files(SRC_PATH, DST_PATH)

print("\n================= DONE =================")
print(f"Copied: ok={cp_ok}, fail={cp_fail}")
print("Destination items now:", len(safe_ls(DST_PATH)))

StatementMeta(, 069bf762-aacd-4bc2-830f-8a5c8f6996b4, 12, Finished, Available, Finished, False)

SRC_PATH = abfss://b13d87d7-076e-42df-a992-14dbd7f186f2@onelake.dfs.fabric.microsoft.com/da429e12-11e6-46fb-9769-53af4ec87b21/Files/Backup_asof_2025-12-31
DST_PATH = abfss://b13d87d7-076e-42df-a992-14dbd7f186f2@onelake.dfs.fabric.microsoft.com/cae086eb-7751-49db-87cd-e57f036d9d05/Files/RealPageDaily

🧹 Cleaning destination ...
Deleted entries: ok=1, fail=0
Destination items now: 0

📦 Copying backup -> destination ...
  ...copied 50
  ...copied 100
  ...copied 150
  ...copied 200
  ...copied 250
  ...copied 300
  ...copied 350
  ...copied 400

================= DONE =================
Copied: ok=439, fail=0
Destination items now: 439



# **Files---->stg---->historic tables**


In [11]:
# ======================= FABRIC PySpark: SCD2 BLINDADO (FULL) =======================
# ✅ ALWAYS:
#   1) READ CSV from Files/RealPageDaily using tables array (case-insensitive)
#   2) BUILD STG with HASH (hash_value) EVERY RUN  ✅ <--- FIX REAL
#   3) OVERWRITE stg_<table> EVERY RUN
#   4) RUN SCD2 using that stg_<table>
# ================================================================================

import re
from datetime import timedelta, date
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, DateType, TimestampType
from pyspark.sql.window import Window

# -------------------- CONSTANTS --------------------
HASH_COL = "hash_value"
START_COL = "Start_Date"
END_COL   = "End_Date"
ISCURRENT_COL = "IsCurrent"

MAX_END = "12/31/9999"
DEFAULT_START = "1/1/2025"

TODAY = date.today()
YESTERDAY = TODAY - timedelta(days=1)
TODAY_STR = f"{TODAY.month}/{TODAY.day}/{TODAY.year}"
YEST_STR  = f"{YESTERDAY.month}/{YESTERDAY.day}/{YESTERDAY.year}"

spark.conf.set("spark.sql.legacy.timeParserPolicy", "CORRECTED")

DATE_PATTERNS = [
    "M/d/yyyy","MM/dd/yyyy","M/d/yy","MM/dd/yy",
    "yyyy-MM-dd","yyyy/MM/dd","yyyyMMdd",
    "dd-MMM-yyyy","d-MMM-yyyy","MMM d, yyyy","MMMM d, yyyy"
]
TS_PATTERNS = [
    "M/d/yyyy h:mm:ss a","MM/dd/yyyy h:mm:ss a",
    "M/d/yyyy hh:mm:ss a","MM/dd/yyyy hh:mm:ss a",
    "M/d/yyyy H:mm:ss","MM/dd/yyyy H:mm:ss",
    "M/d/yyyy HH:mm:ss","MM/dd/yyyy HH:mm:ss",
    "yyyy-MM-dd HH:mm:ss","yyyy-MM-dd'T'HH:mm:ss","yyyy-MM-dd'T'HH:mm:ss.SSS"
]

# -------------------- HELPERS --------------------
def _strip_stg(name: str) -> str:
    return re.sub(r"^stg_", "", (name or "").strip(), flags=re.I)

def _parent_child_names(name_in_array: str):
    base = _strip_stg(name_in_array)
    return base, f"stg_{base}"

def ensure_dates_cols(df):
    if START_COL not in df.columns:
        df = df.withColumn(START_COL, F.lit(DEFAULT_START))
    if END_COL not in df.columns:
        df = df.withColumn(END_COL, F.lit(MAX_END))
    return df

def ensure_iscurrent(df):
    if ISCURRENT_COL not in df.columns:
        df = df.withColumn(ISCURRENT_COL, F.lit(1))
    return df

def _has_data(df):
    return df.limit(1).count() > 0

def _parsed_date_expr(col_trim):
    parsed_ts = F.coalesce(*[F.to_timestamp(col_trim, p) for p in TS_PATTERNS])
    parsed_d  = F.coalesce(*[F.to_date(col_trim, p) for p in DATE_PATTERNS])
    return F.coalesce(F.to_date(parsed_ts), parsed_d)

def standardize_date_columns(df, sample_rows=1000, success_threshold=0.90, min_non_null=10):
    out = df
    sample = df.limit(sample_rows)

    for field in df.schema.fields:
        c = field.name
        dtp = field.dataType

        if isinstance(dtp, (DateType, TimestampType)):
            out = out.withColumn(c, F.date_format(F.to_date(F.col(c)), "M/dd/yyyy"))
            continue

        if isinstance(dtp, StringType):
            col_trim = F.trim(F.col(c))
            parsed_date = _parsed_date_expr(col_trim)

            stats = (
                sample.select(
                    F.sum(F.when(col_trim.isNotNull() & (F.length(col_trim) > 0), 1).otherwise(0)).alias("non_null"),
                    F.sum(F.when((col_trim.isNotNull() & (F.length(col_trim) > 0)) & parsed_date.isNotNull(), 1).otherwise(0)).alias("parsed_ok")
                ).collect()[0]
            )

            non_null = int(stats["non_null"] or 0)
            parsed_ok = int(stats["parsed_ok"] or 0)
            rate = (parsed_ok / non_null) if non_null > 0 else 0.0

            if non_null >= min_non_null and rate >= success_threshold:
                out = out.withColumn(c, F.date_format(parsed_date, "M/dd/yyyy"))

    return out

def _norm_key(s: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", (s or "").lower().strip())

def normalize_excludes(df, exclude_cols):
    cols_map = {_norm_key(c): c for c in df.columns}
    out = []
    for x in (exclude_cols or []):
        k = _norm_key(str(x))
        if k in cols_map:
            out.append(cols_map[k])
    return out

# ✅ CHANGED: add_hash now supports include_cols (whitelist) OR exclude_cols (blacklist)
def add_hash(df, exclude_cols=None, include_cols=None):
    if include_cols:
        # Whitelist mode: use ONLY the specified columns (still sorted for determinism)
        include_set = set(include_cols)
        cols_for_hash = [c for c in df.columns if c in include_set]
        cols_for_hash = sorted(cols_for_hash, key=lambda x: x.lower())
    else:
        # Blacklist mode (default behavior): exclude the specified columns
        exclude_set = set((exclude_cols or []) + [HASH_COL, START_COL, END_COL, ISCURRENT_COL])
        cols_for_hash = [c for c in df.columns if c not in exclude_set]
        cols_for_hash = sorted(cols_for_hash, key=lambda x: x.lower())

    if not cols_for_hash:
        return df.withColumn(HASH_COL, F.sha2(F.lit(""), 256))

    exprs = [F.coalesce(F.trim(F.col(c).cast("string")), F.lit("")) for c in cols_for_hash]
    return df.withColumn(HASH_COL, F.sha2(F.concat_ws("||", *exprs), 256))

def _order_ts(df, colname):
    if colname not in df.columns:
        return None
    col_trim = F.trim(F.col(colname).cast("string"))
    d = _parsed_date_expr(col_trim)
    return F.to_timestamp(d)

def dedup_child_latest(df_child, bk_cols):
    order_cols = ["RecordModifiedDate","RecordCreatedDate","CDSExtractDate","osl_ExtractDate","CreateDate","RowStartDate","ModifyDate","PostDate"]
    order_exprs = []
    for c in order_cols:
        ts_expr = _order_ts(df_child, c)
        if ts_expr is not None:
            order_exprs.append(ts_expr.desc_nulls_last())

    if HASH_COL in df_child.columns:
        order_exprs.append(F.col(HASH_COL).desc_nulls_last())
    if "SourceKey" in df_child.columns:
        order_exprs.append(F.col("SourceKey").desc_nulls_last())
    for c in bk_cols:
        if c in df_child.columns:
            order_exprs.append(F.col(c).cast("string").desc_nulls_last())

    if not order_exprs:
        return df_child.dropDuplicates(bk_cols)

    w = Window.partitionBy(*[F.col(c) for c in bk_cols]).orderBy(*order_exprs)
    return (df_child
            .withColumn("_rn", F.row_number().over(w))
            .where(F.col("_rn") == 1)
            .drop("_rn"))

def consolidate_unique_hash_ranges(df_parent_out, bk_cols):
    start_p = _parsed_date_expr(F.trim(F.col(START_COL).cast("string")))
    end_p = F.when(F.trim(F.col(END_COL)) == F.lit(MAX_END),
                   F.to_date(F.lit(MAX_END), "M/d/yyyy")) \
             .otherwise(_parsed_date_expr(F.trim(F.col(END_COL).cast("string"))))

    df = (df_parent_out
          .withColumn("_start_p", start_p)
          .withColumn("_end_p", end_p))

    w = Window.partitionBy(*[F.col(c) for c in (bk_cols + [HASH_COL])])
    df = (df
          .withColumn("_start_min", F.min(F.col("_start_p")).over(w))
          .withColumn("_end_max",   F.max(F.col("_end_p")).over(w)))

    w_first = Window.partitionBy(*[F.col(c) for c in (bk_cols + [HASH_COL])]).orderBy(F.col("_start_p").asc_nulls_last())
    df = (df
          .withColumn("_rn_keep", F.row_number().over(w_first))
          .where(F.col("_rn_keep") == 1)
          .drop("_rn_keep"))

    df = df.withColumn(START_COL, F.date_format(F.col("_start_min"), "M/dd/yyyy"))
    df = df.withColumn(
        END_COL,
        F.when(F.col("_end_max") == F.to_date(F.lit(MAX_END), "M/d/yyyy"), F.lit(MAX_END))
         .otherwise(F.date_format(F.col("_end_max"), "M/dd/yyyy"))
    )

    start2 = _parsed_date_expr(F.trim(F.col(START_COL).cast("string")))
    end2 = F.when(F.trim(F.col(END_COL)) == F.lit(MAX_END),
                  F.to_date(F.lit(MAX_END), "M/d/yyyy")) \
            .otherwise(_parsed_date_expr(F.trim(F.col(END_COL).cast("string"))))

    wbk = Window.partitionBy(*[F.col(c) for c in bk_cols]).orderBy(start2.asc_nulls_last())
    next_start = F.lead(start2).over(wbk)

    end_fix = F.when(
        (F.trim(F.col(END_COL)) == F.lit(MAX_END)) & next_start.isNotNull(),
        F.date_sub(next_start, 1)
    ).otherwise(end2)

    end_fix = F.greatest(start2, end_fix)

    df = (df
          .withColumn("_end_fix", end_fix)
          .withColumn(
              END_COL,
              F.when(F.col("_end_fix") == F.to_date(F.lit(MAX_END), "M/d/yyyy"), F.lit(MAX_END))
               .otherwise(F.date_format(F.col("_end_fix"), "M/dd/yyyy"))
          )
          .drop("_start_p","_end_p","_start_min","_end_max","_end_fix"))

    df = df.withColumn(ISCURRENT_COL, F.when(F.trim(F.col(END_COL)) == F.lit(MAX_END), F.lit(1)).otherwise(F.lit(0)))
    return df

def force_single_current_per_bk(df, bk_cols):
    start_d = _parsed_date_expr(F.trim(F.col(START_COL).cast("string")))
    end_d = F.when(F.trim(F.col(END_COL)) == F.lit(MAX_END),
                   F.to_date(F.lit(MAX_END), "M/d/yyyy")) \
             .otherwise(_parsed_date_expr(F.trim(F.col(END_COL).cast("string"))))

    w = Window.partitionBy(*[F.col(c) for c in bk_cols]).orderBy(end_d.desc_nulls_last(), start_d.desc_nulls_last())
    df2 = df.withColumn("_rn_curr", F.row_number().over(w))

    return (df2
        .withColumn(END_COL, F.when(F.col("_rn_curr") == 1, F.lit(MAX_END)).otherwise(F.col(END_COL)))
        .withColumn(ISCURRENT_COL, F.when(F.col("_rn_curr") == 1, F.lit(1)).otherwise(F.lit(0)))
        .drop("_rn_curr")
    )

# -------------------- RAW -> STG (with HASH) --------------------
RAW_DIR = "Files/RealPageDaily"

def _norm_name(s: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", (s or "").lower().strip())

def _list_files(path: str):
    try:
        return mssparkutils.fs.ls(path)
    except Exception:
        return dbutils.fs.ls(path)

def _pick_csv_paths_for_table(arr_name: str):
    want = _norm_name(arr_name)
    items = _list_files(RAW_DIR)
    matches = []

    for it in items:
        fname = getattr(it, "name", None) or getattr(it, "path", "").split("/")[-1]
        fpath = getattr(it, "path", None) or (RAW_DIR.rstrip("/") + "/" + fname)

        stem = re.sub(r"\.csv$", "", fname, flags=re.I)
        got = _norm_name(stem)

        if got == want:
            matches.append(fpath)

    return matches

def _read_raw_csv_for_table(arr_name: str):
    paths = _pick_csv_paths_for_table(arr_name)
    if not paths:
        raise ValueError(f"No encontré CSV en {RAW_DIR} para {arr_name}")
    return (spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(paths)
    )

def _try_table_ci(tbl_name: str):
    try:
        return spark.table(tbl_name)
    except Exception:
        pass

    db = spark.catalog.currentDatabase()
    want = (tbl_name or "").strip().lower()
    for t in spark.catalog.listTables(db):
        if (t.name or "").strip().lower() == want:
            return spark.table(t.name)
    raise

# ✅ CHANGED: resolves include_hash_cols vs exclude_hash_cols before calling add_hash
def _resolve_hash_kwargs(df, table_cfg: dict) -> dict:
    incl_cfg = table_cfg.get("include_hash_cols")
    if incl_cfg:
        incl_norm = normalize_excludes(df, incl_cfg)  # reuses case-insensitive mapper
        return {"include_cols": incl_norm}
    excl_cfg = table_cfg.get("exclude_hash_cols", [])
    excl_norm = normalize_excludes(df, excl_cfg)
    return {"exclude_cols": excl_norm}

def _overwrite_stg_from_raw_with_hash(table_cfg: dict):
    name_in = table_cfg["name"]
    _, child_tbl = _parent_child_names(name_in)

    df_raw = _read_raw_csv_for_table(name_in)

    # ✅ Normalize dates (same rules)
    df_raw = standardize_date_columns(df_raw)

    # ✅ Ensure technical cols exist (optional but nice for STG inspection)
    df_raw = ensure_iscurrent(ensure_dates_cols(df_raw))

    # ✅ ADD HASH ALWAYS (include or exclude, depending on config)
    df_stg = add_hash(df_raw, **_resolve_hash_kwargs(df_raw, table_cfg))

    # ✅ WRITE STG WITH HASH (overwrite)
    (df_stg.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(child_tbl)
    )

    print(f"🧱 STG overwritten WITH hash: {child_tbl} | rows={df_stg.count()}")

def _load_parent_or_empty(parent_tbl: str, child_df_schema_ref):
    try:
        return _try_table_ci(parent_tbl)
    except Exception:
        print(f"⚠ No existe {parent_tbl}. Creando parent vacío (primer run).")
        return spark.createDataFrame([], child_df_schema_ref.schema)

# -------------------- MAIN --------------------
def scd2_update_parent_from_child(table_cfg: dict):
    name_in = table_cfg["name"]
    bk_cols = table_cfg.get("business_key", [])
    if not bk_cols:
        raise ValueError(f"{name_in}: business_key vacío.")

    parent_tbl, child_tbl = _parent_child_names(name_in)
    print(f"🔎 Parent: {parent_tbl} | Child: {child_tbl}")

    # ✅ ALWAYS: RAW -> STG WITH HASH (every run)
    _overwrite_stg_from_raw_with_hash(table_cfg)

    # ✅ Use STG (already has hash_value)
    df_child_raw  = _try_table_ci(child_tbl)
    df_parent_raw = _load_parent_or_empty(parent_tbl, df_child_raw)

    # ---- Keep your logic: parent may need standardize + ensure tech cols ----
    df_parent_raw = standardize_date_columns(df_parent_raw)
    df_parent     = ensure_iscurrent(ensure_dates_cols(df_parent_raw))

    # ✅ CHANGED: Parent hash also respects include/exclude from config
    df_parent = add_hash(df_parent, **_resolve_hash_kwargs(df_parent, table_cfg))

    # Guard BK
    miss_c = [c for c in bk_cols if c not in df_child_raw.columns]
    miss_p = [c for c in bk_cols if c not in df_parent.columns]
    if miss_c: raise ValueError(f"{child_tbl}: falta BK {miss_c}")
    if miss_p: raise ValueError(f"{parent_tbl}: falta BK {miss_p}")

    # Dedup child (already has hash)
    df_child = dedup_child_latest(df_child_raw, bk_cols).cache()
    _ = df_child.count()

    c  = df_child.alias("c")
    pc = df_parent.where(F.trim(F.col(END_COL)) == F.lit(MAX_END)).alias("pc")
    ph = df_parent.where(F.trim(F.col(END_COL)) != F.lit(MAX_END)).alias("ph")

    cond_cp = [F.col(f"c.{x}").cast("string") == F.col(f"pc.{x}").cast("string") for x in bk_cols]

    changed_keys = (c
        .join(pc, on=cond_cp, how="inner")
        .where(F.col(f"c.{HASH_COL}") != F.col(f"pc.{HASH_COL}"))
        .select([F.col(f"c.{x}").alias(x) for x in bk_cols])
        .dropDuplicates(bk_cols)
    )

    new_bk_rows = (c
        .join(pc, on=cond_cp, how="left_anti")
        .select("c.*")
        .dropDuplicates(bk_cols)
    )

    to_close = None
    still_curr_ok = pc.select("pc.*")

    if _has_data(changed_keys):
        k = changed_keys.alias("k")
        cond_pk = [F.col(f"pc.{x}").cast("string") == F.col(f"k.{x}").cast("string") for x in bk_cols]

        start_p = _parsed_date_expr(F.trim(F.col(f"pc.{START_COL}").cast("string")))
        yest_p  = _parsed_date_expr(F.lit(YEST_STR))
        today_p = _parsed_date_expr(F.lit(TODAY_STR))
        safe_end = F.least(today_p, F.greatest(start_p, yest_p))

        to_close = (pc
            .join(k, on=cond_pk, how="inner")
            .select("pc.*")
            .withColumn(END_COL, F.date_format(safe_end, "M/dd/yyyy"))
        )

        still_curr_ok = (pc
            .join(k, on=cond_pk, how="left_anti")
            .select("pc.*")
        )

    inserts = []
    if _has_data(new_bk_rows):
        inserts.append(new_bk_rows
            .withColumn(START_COL, F.lit(TODAY_STR))
            .withColumn(END_COL, F.lit(MAX_END))
        )

    changed_new = None
    if _has_data(changed_keys):
        k2 = changed_keys.alias("k2")
        cond_ck = [F.col(f"c.{x}").cast("string") == F.col(f"k2.{x}").cast("string") for x in bk_cols]
        child_changed = (c.join(k2, on=cond_ck, how="inner")
                         .select("c.*")
                         .dropDuplicates(bk_cols))

        if _has_data(child_changed):
            changed_new = (child_changed
                .withColumn(START_COL, F.lit(TODAY_STR))
                .withColumn(END_COL, F.lit(MAX_END))
            )

    df_parent_out = ph.unionByName(still_curr_ok, allowMissingColumns=True)
    if to_close is not None:
        df_parent_out = df_parent_out.unionByName(to_close, allowMissingColumns=True)
    if inserts:
        for ins in inserts:
            df_parent_out = df_parent_out.unionByName(ins, allowMissingColumns=True)
    if changed_new is not None:
        df_parent_out = df_parent_out.unionByName(changed_new, allowMissingColumns=True)

    df_parent_out = standardize_date_columns(df_parent_out)
    df_parent_out = consolidate_unique_hash_ranges(df_parent_out, bk_cols)
    df_parent_out = force_single_current_per_bk(df_parent_out, bk_cols)

    (df_parent_out.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(parent_tbl)
    )

    df_child.unpersist()
    print(f"✅ OK: {child_tbl} -> {parent_tbl} | SCD2 blindado")

# -------------------- RUN --------------------
print("🚀 Starting SCD2 blindado run...")
print("Today:", TODAY_STR, "| Yesterday:", YEST_STR)
print("Tables in array:", len(tables))

failed = []
for cfg in tables:
    nm = cfg.get("name", "unknown")
    try:
        print(f"\n================= {nm} =================")
        scd2_update_parent_from_child(cfg)
        print(f"✅ DONE: {nm}")
    except Exception as e:
        failed.append((nm, str(e)))
        print(f"❌ FAIL: {nm}: {str(e)[:600]}")

print("\n🎯 FINISH")
if failed:
    print("⚠ Tablas fallidas:")
    for n, err in failed:
        print("  •", n, "|", err[:600])
else:
    print("✅ Todo OK.")

StatementMeta(, 837beadd-fd7c-4962-8796-24ec86d89df3, 13, Finished, Available, Finished, False)

🚀 Starting SCD2 blindado run...
Today: 3/5/2026 | Yesterday: 3/4/2026
Tables in array: 21

================= dimaccountgrouphierarchy__0001 =================
🔎 Parent: dimaccountgrouphierarchy__0001 | Child: stg_dimaccountgrouphierarchy__0001
🧱 STG overwritten WITH hash: stg_dimaccountgrouphierarchy__0001 | rows=272
✅ OK: stg_dimaccountgrouphierarchy__0001 -> dimaccountgrouphierarchy__0001 | SCD2 blindado
✅ DONE: dimaccountgrouphierarchy__0001

================= dimclassificationlist__0001 =================
🔎 Parent: dimclassificationlist__0001 | Child: stg_dimclassificationlist__0001
🧱 STG overwritten WITH hash: stg_dimclassificationlist__0001 | rows=0
✅ OK: stg_dimclassificationlist__0001 -> dimclassificationlist__0001 | SCD2 blindado
✅ DONE: dimclassificationlist__0001

================= dimleadmanagement__0001 =================
🔎 Parent: dimleadmanagement__0001 | Child: stg_dimleadmanagement__0001
🧱 STG overwritten WITH hash: stg_dimleadmanagement__0001 | rows=24137
✅ OK: stg_dimle

In [4]:
from pyspark.sql import functions as F

tbl = "dimservicerequest__0001"
bk_col = "ServiceRequestKey"
bk_val = 34573

df = (spark.table(tbl)
      .filter(F.col(bk_col) == F.lit(bk_val))
      .orderBy(F.col("Start_Date").desc())   # opcional si es SCD2
)

display(df)          # ✅ sale en tabla (no en texto)
# df.display()       # alternativa en algunos notebooks

StatementMeta(, 846c89fe-baed-4787-8cb9-ce3d38f4d294, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5fe37792-e725-4751-9ab8-be7c0f1f1b78)

# **dimservicerequest__0001 ADD []**

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


def parse_date_safe(col_):
    s = F.trim(col_.cast("string"))

    s_norm = F.when(
        s.rlike(r"^\d{1,2}/\d{1,2}/\d{4}$"),
        F.concat_ws(
            "/",
            F.lpad(F.split(s, "/").getItem(0), 2, "0"),
            F.lpad(F.split(s, "/").getItem(1), 2, "0"),
            F.split(s, "/").getItem(2),
        ),
    ).otherwise(s)

    return F.coalesce(
        F.to_date(s, "yyyy-MM-dd"),
        F.to_date(s, "yyyy-MM-dd HH:mm:ss"),
        F.to_date(s, "yyyy-MM-dd'T'HH:mm:ss"),
        F.to_date(s_norm, "MM/dd/yyyy"),
        F.to_date(s_norm, "MM/dd/yyyy HH:mm:ss"),
        F.to_date(s_norm, "dd/MM/yyyy"),
        F.to_date(s_norm, "dd/MM/yyyy HH:mm:ss"),
    )


def add_columns(
    df_sr,
    df_date,
    df_mr,
    create_col="CreateDate",
    complete_col="ActualCompletedDate",
    date_col="Date",
    nonwork_col="NonWorkDays",
):
    # -------- Calendar counters --------
    cal = (
        df_date.select(
            F.col(date_col).cast("date").alias("CalDate"),
            F.col(nonwork_col).cast("int").alias("NonWorkDays"),
        )
        .filter(F.col("CalDate").isNotNull())
        .dropDuplicates(["CalDate"])
        .withColumn(
            "IsWorkDay",
            F.when(F.col("NonWorkDays") == 0, F.lit(1)).otherwise(F.lit(0)),
        )
    )

    w = Window.orderBy("CalDate").rowsBetween(Window.unboundedPreceding, Window.currentRow)

    cal = (
        cal.withColumn("CumWorkDays", F.sum("IsWorkDay").over(w))
        .withColumn("WorkDaysBefore", F.col("CumWorkDays") - F.col("IsWorkDay"))
        .select("CalDate", "CumWorkDays", "WorkDaysBefore")
    )

    cal_s = cal.select(
        F.col("CalDate").alias("_StartCal"),
        F.col("WorkDaysBefore").alias("_BeforeStart"),
    )

    cal_e = cal.select(
        F.col("CalDate").alias("_EndCal"),
        F.col("CumWorkDays").alias("_CumEnd"),
    )

    # -------- Base SR dates --------
    sr = df_sr.withColumn(
        "_EndDate",
        F.coalesce(parse_date_safe(F.col(complete_col)), F.current_date()),
    ).withColumn("_StartDate", parse_date_safe(F.col(create_col)))

    # -------- WorkingDays (CreateDate -> EndDate) --------
    sr = (
        sr.join(cal_s, sr["_StartDate"] == cal_s["_StartCal"], "left")
        .join(cal_e, sr["_EndDate"] == cal_e["_EndCal"], "left")
        .withColumn(
            "WorkingDays",
            F.when(
                F.col("_StartDate").isNull()
                | F.col("_EndDate").isNull()
                | F.col("_BeforeStart").isNull()
                | F.col("_CumEnd").isNull()
                | (F.col("_EndDate") < F.col("_StartDate")),
                F.lit(None).cast("int"),
            ).otherwise(
                F.greatest(
                    (F.col("_CumEnd") - F.col("_BeforeStart")) - F.lit(1), F.lit(0)
                ).cast("int")
            ),
        )
        .drop("_StartCal", "_EndCal", "_BeforeStart", "_CumEnd")
    )

    # -------- Lookup MakeReadyStartDay + MoveOutDate -> desde df_mr --------
    # ✅ Traemos AMBAS columnas en un solo join para evitar duplicar el join
    mr = (
        df_mr.select(
            "ServiceRequestKey",
            F.col("mrrMakeReadyStartDay").alias("_mrStartDay"),
            F.col("mrrMoveOutDate").alias("_mrMoveOutDate"),  # 👈 NUEVA columna
        ).dropDuplicates(["ServiceRequestKey"])
    )

    sr = (
        sr.join(mr, on="ServiceRequestKey", how="left")
        # --- columna existente ---
        .withColumn(
            "mrrrMakeReadyStartDay",
            F.when(
                F.upper(F.trim(F.col("ServiceRequestTypeCode"))) == F.lit("MR"),
                F.col("_mrStartDay"),
            ).otherwise(F.lit(None)),
        )
        # --- ✅ NUEVA columna: mrrrMoveOutDate (equivalente al DAX) ---
        .withColumn(
            "mrrrMoveOutDate",
            F.when(
                F.upper(F.trim(F.col("ServiceRequestTypeCode"))) == F.lit("MR"),
                F.col("_mrMoveOutDate"),
            ).otherwise(F.lit(None)),
        )
        .drop("_mrStartDay", "_mrMoveOutDate")
    )

    # -------- MRWorkingDays (mrrrMakeReadyStartDay -> EndDate) --------
    sr = sr.withColumn("_MRStartDate", parse_date_safe(F.col("mrrrMoveOutDate")))

    cal_s_mr = cal_s.select(
        F.col("_StartCal").alias("_MRStartCal"),
        F.col("_BeforeStart").alias("_MRBeforeStart"),
    )
    cal_e_mr = cal_e.select(
        F.col("_EndCal").alias("_MREndCal"),
        F.col("_CumEnd").alias("_MRCumEnd"),
    )

    sr = (
        sr.join(cal_s_mr, sr["_MRStartDate"] == cal_s_mr["_MRStartCal"], "left")
        .join(cal_e_mr, sr["_EndDate"] == cal_e_mr["_MREndCal"], "left")
        .withColumn(
            "MRWorkingDays",
            F.when(
                F.col("_MRStartDate").isNull()
                | F.col("_EndDate").isNull()
                | F.col("_MRBeforeStart").isNull()
                | F.col("_MRCumEnd").isNull()
                | (F.col("_EndDate") < F.col("_MRStartDate")),
                F.lit(None).cast("int"),
            ).otherwise(
                F.greatest(
                    (F.col("_MRCumEnd") - F.col("_MRBeforeStart")) - F.lit(1), F.lit(0)
                ).cast("int")
            ),
        )
        .drop(
            "_MRStartCal", "_MREndCal", "_MRBeforeStart",
            "_MRCumEnd", "_MRStartDate", "_EndDate", "_StartDate",
        )
    )

    return sr


# ===== USO =====
df_sr   = spark.read.table("dimservicerequest__0001")
df_date = spark.read.table("Date")
df_mr   = spark.read.table("dimmakereadyrequest__0001")

out = add_columns(df_sr, df_date, df_mr)

out.select(
    "ServiceRequestKey",
    "ServiceRequestTypeCode",
    "WorkingDays",
    "mrrrMakeReadyStartDay",
    "MRWorkingDays",
    "mrrrMoveOutDate",  # 👈 nueva columna en el preview
).show(30, truncate=False)

(
    out
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable("dimservicerequest__0001")
)


StatementMeta(, 5df60462-d0d3-4cc3-a1ea-d4ea5a91555b, 6, Finished, Available, Finished, False)

+-----------------+----------------------+-----------+---------------------+-------------+---------------+
|ServiceRequestKey|ServiceRequestTypeCode|WorkingDays|mrrrMakeReadyStartDay|MRWorkingDays|mrrrMoveOutDate|
+-----------------+----------------------+-----------+---------------------+-------------+---------------+
|2                |RWO                   |NULL       |NULL                 |NULL         |NULL           |
|18               |1                     |NULL       |NULL                 |NULL         |NULL           |
|30               |1                     |NULL       |NULL                 |NULL         |NULL           |
|36               |1                     |NULL       |NULL                 |NULL         |NULL           |
|42               |MR                    |NULL       |8/07/2018            |NULL         |8/06/2018      |
|46               |1                     |NULL       |NULL                 |NULL         |NULL           |
|66               |RWO               

# **Duplicate tables**

In [3]:
from collections import Counter
from pyspark.sql import functions as F

def make_unique_columns(df):

    seen = Counter()
    new_cols = []
    for c in df.columns:
        key = c.lower()
        seen[key] += 1
        if seen[key] == 1:
            new_cols.append(c)
        else:
            new_cols.append(f"{c}__{seen[key]}")
    return df.toDF(*new_cols)

def duplicate_table(src_table, dst_table, db=None):
    src = f"{db}.{src_table}" if db else src_table
    dst = f"{db}.{dst_table}" if db else dst_table

    df = spark.table(src)

    df = make_unique_columns(df)


    spark.sql(f"DROP TABLE IF EXISTS {dst}")


    (df.write
      .format("delta")
      .mode("overwrite")
      .option("overwriteSchema", "true")
      .saveAsTable(dst)
    )

    print(f"✅ Duplicated {src} -> {dst} | cols={len(df.columns)} | rows={df.count()}")


CURRENT_DB = spark.sql("select current_database() as db").collect()[0]["db"]

duplicate_table("dimleaseexpiration__0001", "dimleaseexpiration__0001_2", db=CURRENT_DB)
duplicate_table("dimleaseexpiration__0001", "dimleaseexpiration__0001_3", db=CURRENT_DB)
duplicate_table("dimservicerequest__0001", "dimservicerequest__0001_2", db=CURRENT_DB)

print("Tables duplicated ✅")

StatementMeta(, 2f59edca-3248-4b1f-8edc-8585372000ed, 5, Finished, Available, Finished, False)

✅ Duplicated realpagebix__lah_df2.dimleaseexpiration__0001 -> realpagebix__lah_df2.dimleaseexpiration__0001_2 | cols=78 | rows=9469
✅ Duplicated realpagebix__lah_df2.dimleaseexpiration__0001 -> realpagebix__lah_df2.dimleaseexpiration__0001_3 | cols=78 | rows=9469
✅ Duplicated realpagebix__lah_df2.dimservicerequest__0001 -> realpagebix__lah_df2.dimservicerequest__0001_2 | cols=50 | rows=49622
Tables duplicated ✅


In [2]:
df = (spark.table("RealPageBIX__LAH_DF2.dimservicerequest__0001")
        .orderBy("PropertyKey")   # A → Z
        .limit(1000)
     )

display(df)

StatementMeta(, 9af0fe90-4249-4945-b1fd-108bee9a1c4e, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4761372b-70ac-467d-94f0-13af762ee5f0)